In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

import sys
from pathlib import Path
import pandas as pd
sys.path.append('..')


In [2]:
# =========================================================
# PATHS
# =========================================================

MODEL_PATH = Path("../models/final_ml_model.pkl")

# Burayı kendi final processed dataframe dosyanın adıyla eşleştir
DATA_PATH = Path("../data/processed/main/train.parquet")

FORECAST_DIR = Path("../data/processed/forecast")
FORECAST_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_PATH = FORECAST_DIR / "future_forecast_14d.parquet"


# =========================================================
# LOAD
# =========================================================

model = joblib.load(MODEL_PATH)

df = pd.read_parquet(DATA_PATH)

df["Date"] = pd.to_datetime(df["Date"])

df = df.sort_values(
    ["Store ID", "Product ID", "Date"]
).reset_index(drop=True)

print("Model:", type(model).__name__)
print("Data shape:", df.shape)
print("Date:", df["Date"].min(), "->", df["Date"].max())

Model: LGBMRegressor
Data shape: (66900, 40)
Date: 2022-01-01 00:00:00 -> 2023-10-31 00:00:00


In [3]:
MODEL_FEATURES = [
    "Store_ID",
    "Product_ID",
    "Inventory_Level",
    "Price",
    "Discount",
    "Holiday/Promotion",
    "Year",
    "Month",
    "Day",
    "DayOfWeek",
    "NET_PRICE",
    "inventory_lag_1",
    "is_out_of_stock_lag_1",
    "Price_Diff",
    "Price_Ratio",
    "units_sold_lag_1",
    "units_sold_lag_2",
    "units_sold_lag_3",
    "units_sold_lag_7",
    "units_sold_lag_14",
    "units_sold_lag_30",
    "units_sold_lag_365",
    "sales_roll_mean_7",
    "sales_roll_mean_30",
    "Category_Electronics",
    "Category_Furniture",
    "Category_Groceries",
    "Category_Toys",
    "Region_North",
    "Region_South",
    "Region_West",
    "Weather_Condition_Rainy",
    "Weather_Condition_Snowy",
    "Weather_Condition_Sunny",
    "Seasonality_NEW_Spring",
    "Seasonality_NEW_Summer",
    "Seasonality_NEW_Winter"
]

print("Feature count:", len(MODEL_FEATURES))

Feature count: 37


In [4]:
FORECAST_DAYS = 14

last_date = df["Date"].max()

future_dates = pd.date_range(
    start=last_date + pd.Timedelta(days=1),
    periods=FORECAST_DAYS,
    freq="D"
)

stores = df["Store ID"].unique()
products = df["Product ID"].unique()

print("Last date:", last_date)
print("Forecast:", future_dates[0], "->", future_dates[-1])

print("Stores:", len(stores))
print("Products:", len(products))
print("Expected rows:", len(stores) * len(products) * FORECAST_DAYS)

Last date: 2023-10-31 00:00:00
Forecast: 2023-11-01 00:00:00 -> 2023-11-14 00:00:00
Stores: 5
Products: 20
Expected rows: 1400


In [5]:
df.columns
df.head(5)

,Date,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Price,Discount,Holiday/Promotion,Year,...,Category_Toys,Region_North,Region_South,Region_West,Weather Condition_Rainy,Weather Condition_Snowy,Weather Condition_Sunny,Seasonality_NEW_Spring,Seasonality_NEW_Summer,Seasonality_NEW_Winter
0,2022-01-01,S001,P0001,231,127,55,33.50,20,0,2022,...,0,1,0,0,1,0,0,0,0,1
1,2022-01-02,S001,P0001,116,81,104,27.95,10,0,2022,...,0,0,0,1,0,0,0,0,0,1
2,2022-01-03,S001,P0001,154,5,189,62.70,20,0,2022,...,0,0,0,1,1,0,0,0,0,1
3,2022-01-04,S001,P0001,85,58,193,77.88,15,1,2022,...,0,0,1,0,0,0,0,0,0,1
4,2022-01-05,S001,P0001,238,147,37,28.46,20,1,2022,...,0,0,1,0,0,0,1,0,0,1


In [6]:
history = df.copy()

forecast_results = []

In [7]:
def get_series(history, store_id, product_id):

    series = history[
        (history["Store ID"] == store_id) &
        (history["Product ID"] == product_id)
    ].sort_values("Date").copy()

    return series

In [8]:
def safe_last(series, column):
    """
    Kolonun son bilinen değerini döndürür.
    """

    if column not in series.columns:
        return np.nan

    values = series[column].dropna()

    if len(values) == 0:
        return np.nan

    return values.iloc[-1]

In [9]:
def safe_lag(series, column, lag):
    """
    Geçmişteki lag değerini döndürür.

    lag=1 -> son gözlem
    lag=2 -> iki gözlem önce
    """

    if column not in series.columns:
        return np.nan

    values = series[column].dropna().tolist()

    if len(values) < lag:
        return np.nan

    return values[-lag]

In [10]:
def add_categorical_features(row, series):
    
    # =====================================================
    # CATEGORY
    # =====================================================

    category = safe_last(series, "Category")

    row["Category_Electronics"] = int(
        category == "Electronics"
    )

    row["Category_Furniture"] = int(
        category == "Furniture"
    )

    row["Category_Groceries"] = int(
        category == "Groceries"
    )

    row["Category_Toys"] = int(
        category == "Toys"
    )


    # =====================================================
    # REGION
    # =====================================================

    region = safe_last(series, "Region")

    row["Region_North"] = int(
        region == "North"
    )

    row["Region_South"] = int(
        region == "South"
    )

    row["Region_West"] = int(
        region == "West"
    )


    # =====================================================
    # WEATHER
    # =====================================================

    weather = safe_last(
        series,
        "Weather Condition"
    )

    row["Weather_Condition_Rainy"] = int(
        weather == "Rainy"
    )

    row["Weather_Condition_Snowy"] = int(
        weather == "Snowy"
    )

    row["Weather_Condition_Sunny"] = int(
        weather == "Sunny"
    )


    return row

In [11]:
def add_seasonality(row, date):

    month = date.month

    row["Seasonality_NEW_Spring"] = int(
        month in [3, 4, 5]
    )

    row["Seasonality_NEW_Summer"] = int(
        month in [6, 7, 8]
    )

    row["Seasonality_NEW_Winter"] = int(
        month in [12, 1, 2]
    )

    return row

In [12]:
df.columns

Index(['Date', 'Store ID', 'Product ID', 'Inventory Level', 'Units Sold',
       'Units Ordered', 'Price', 'Discount', 'Holiday/Promotion', 'Year',
       'Month', 'Day', 'DayOfWeek', 'NET_PRICE', 'inventory_lag_1',
       'is_out_of_stock_lag_1', 'Price_Diff', 'Price_Ratio',
       'units_sold_lag_1', 'units_sold_lag_2', 'units_sold_lag_3',
       'units_sold_lag_7', 'units_sold_lag_14', 'units_sold_lag_30',
       'units_sold_lag_365', 'sales_roll_mean_7', 'sales_roll_mean_30',
       'Category_Electronics', 'Category_Furniture', 'Category_Groceries',
       'Category_Toys', 'Region_North', 'Region_South', 'Region_West',
       'Weather Condition_Rainy', 'Weather Condition_Snowy',
       'Weather Condition_Sunny', 'Seasonality_NEW_Spring',
       'Seasonality_NEW_Summer', 'Seasonality_NEW_Winter'],
      dtype='object')

In [ ]:
# =========================================================
# RECURSIVE 14-DAY FORECAST
# =========================================================

for forecast_date in future_dates:

    print(
        f"Forecasting {forecast_date.date()}..."
    )

    for store_id in stores:

        for product_id in products:

            # -------------------------------------------------
            # GET HISTORY
            # -------------------------------------------------

            series = get_series(
                history,
                store_id,
                product_id
            )

            if len(series) == 0:
                continue


            # -------------------------------------------------
            # CREATE ROW
            # -------------------------------------------------

            row = {}

            row["Store_ID"] = store_id
            row["Product_ID"] = product_id


            # -------------------------------------------------
            # DATE FEATURES
            # -------------------------------------------------

            row["Year"] = forecast_date.year
            row["Month"] = forecast_date.month
            row["Day"] = forecast_date.day
            row["DayOfWeek"] = forecast_date.dayofweek


            # -------------------------------------------------
            # CURRENT / FUTURE INVENTORY
            # -------------------------------------------------

            # Son bilinen inventory
            previous_inventory = safe_last(
                series,
                "Inventory_Level"
            )

            # Son tahmini satış
            previous_sales = safe_last(
                series,
                "Units Sold"
            )

            # -------------------------------------------------
            # INVENTORY SIMULATION
            # -------------------------------------------------

            if pd.isna(previous_inventory):

                previous_inventory = 0

            if pd.isna(previous_sales):

                previous_sales = 0


            # Gelecek inventory:
            #
            # mevcut stok - önceki gün satış
            #
            simulated_inventory = max(
                0,
                previous_inventory - previous_sales
            )

            row["Inventory_Level"] = simulated_inventory


            # -------------------------------------------------
            # INVENTORY LAG
            # -------------------------------------------------

            row["inventory_lag_1"] = previous_inventory


            # -------------------------------------------------
            # OUT OF STOCK LAG
            # -------------------------------------------------

            row["is_out_of_stock_lag_1"] = int(
                previous_inventory <= 0
            )


            # -------------------------------------------------
            # PRICE
            # -------------------------------------------------

            row["Price"] = safe_last(
                series,
                "Price"
            )

            row["Discount"] = safe_last(
                series,
                "Discount"
            )


            # -------------------------------------------------
            # HOLIDAY / PROMOTION
            # -------------------------------------------------

            row["Holiday/Promotion"] = safe_last(
                series,
                "Holiday/Promotion"
            )


            # -------------------------------------------------
            # NET PRICE
            # -------------------------------------------------

            price = row["Price"]
            discount = row["Discount"]

            if pd.isna(price):
                price = 0

            if pd.isna(discount):
                discount = 0

            row["NET_PRICE"] = price * (1 - discount)   # discount=20 için price*(-19) gibi absürt negatif değerler

            # -------------------------------------------------
            # PRICE DIFF
            # -------------------------------------------------

            previous_price = safe_last(
                series,
                "Price"
            )

            if pd.isna(previous_price):
                previous_price = price

            row["Price_Diff"] = (
                price - previous_price
            )


            # -------------------------------------------------
            # PRICE RATIO
            # -------------------------------------------------

            if previous_price != 0:

                row["Price_Ratio"] = (
                    price / previous_price
                )

            else:

                row["Price_Ratio"] = 1.0


            # -------------------------------------------------
            # SALES LAGS
            # -------------------------------------------------

            row["units_sold_lag_1"] = safe_lag(
                series,
                "Units Sold",
                1
            )

            row["units_sold_lag_2"] = safe_lag(
                series,
                "Units Sold",
                2
            )

            row["units_sold_lag_3"] = safe_lag(
                series,
                "Units Sold",
                3
            )

            row["units_sold_lag_7"] = safe_lag(
                series,
                "Units Sold",
                7
            )

            row["units_sold_lag_14"] = safe_lag(
                series,
                "Units Sold",
                14
            )

            row["units_sold_lag_30"] = safe_lag(
                series,
                "Units Sold",
                30
            )

            row["units_sold_lag_365"] = safe_lag(
                series,
                "Units Sold",
                365
            )


            # -------------------------------------------------
            # ROLLING MEANS
            # -------------------------------------------------

            sales = (
                series["Units Sold"]
                .dropna()
            )


            row["sales_roll_mean_7"] = (
                sales.tail(7).mean()
            )

            row["sales_roll_mean_30"] = (
                sales.tail(30).mean()
            )


            # -------------------------------------------------
            # CATEGORICAL FEATURES
            # -------------------------------------------------

            row = add_categorical_features(
                row,
                series
            )


            # -------------------------------------------------
            # SEASONALITY
            # -------------------------------------------------

            row = add_seasonality(
                row,
                forecast_date
            )


            # -------------------------------------------------
            # CREATE X
            # -------------------------------------------------

            X_future = pd.DataFrame([row])


            # -------------------------------------------------
            # FEATURE ORDER
            # -------------------------------------------------

            X_future = X_future[
                MODEL_FEATURES
            ]


            # -------------------------------------------------
            # CHECK NaN
            # -------------------------------------------------

            if X_future.isnull().any().any():

                missing = X_future.columns[
                    X_future.isnull().any()
                ].tolist()

                raise ValueError(
                    f"NaN bulundu!\n"
                    f"Date: {forecast_date}\n"
                    f"Store: {store_id}\n"
                    f"Product: {product_id}\n"
                    f"Columns: {missing}"
                )


            # -------------------------------------------------
            # PREDICT
            # -------------------------------------------------
            
            # =========================================================
            # LightGBM categorical feature uyumu
            # =========================================================

            for col in ["Store_ID", "Product_ID"]:
                X_future[col] = X_future[col].astype("category")

                # Modelin eğitim sırasında kullandığı kategorileri al
                train_categories = model.booster_.pandas_categorical[
                    ["Store_ID", "Product_ID"].index(col)
                ]

                X_future[col] = X_future[col].cat.set_categories(train_categories)


            prediction = model.predict(
                X_future
            )[0]


            # Demand negatif olamaz
            prediction = max(
                0,
                float(prediction)
            )


            # -------------------------------------------------
            # SAVE RESULT
            # -------------------------------------------------

            forecast_results.append({

                "Date": forecast_date,
                "Store_ID": store_id,
                "Product_ID": product_id,
                "Forecast Demand": prediction

            })


            # -------------------------------------------------
            # ADD PREDICTION TO HISTORY
            # -------------------------------------------------

            new_row = row.copy()

            new_row["Date"] = forecast_date
            new_row["Units Sold"] = prediction

            history = pd.concat(
                [
                    history,
                    pd.DataFrame([new_row])
                ],
                ignore_index=True
            )


print("Recursive forecast tamamlandı.")

Forecasting 2023-11-01...
Forecasting 2023-11-02...
Forecasting 2023-11-03...
Forecasting 2023-11-04...
Forecasting 2023-11-05...
Forecasting 2023-11-06...
Forecasting 2023-11-07...
Forecasting 2023-11-08...
Forecasting 2023-11-09...
Forecasting 2023-11-10...
Forecasting 2023-11-11...
Forecasting 2023-11-12...
Forecasting 2023-11-13...
Forecasting 2023-11-14...
Recursive forecast tamamlandı.


In [15]:
future_forecast = pd.DataFrame(
    forecast_results
)

future_forecast = future_forecast.sort_values(
    [
        "Date",
        "Store_ID",
        "Product_ID"
    ]
).reset_index(drop=True)

print(
    "Forecast shape:",
    future_forecast.shape
)

display(
    future_forecast.head(20)
)

Forecast shape: (1400, 4)


,Date,Store_ID,Product_ID,Forecast Demand
0,2023-11-01,S001,P0001,31.363328
1,2023-11-01,S001,P0002,30.979846
2,2023-11-01,S001,P0003,31.058589
3,2023-11-01,S001,P0004,31.319429
4,2023-11-01,S001,P0005,31.202163
5,2023-11-01,S001,P0006,31.191352
6,2023-11-01,S001,P0007,30.934707
7,2023-11-01,S001,P0008,31.395417
8,2023-11-01,S001,P0009,31.363328
9,2023-11-01,S001,P0010,31.030186


In [19]:
df.groupby(["Store ID", "Product ID"])["Units Sold"].mean().describe()

count    100.000000
mean     136.435575
std        4.586202
min      126.898356
25%      133.050822
50%      136.701046
75%      139.288490
max      149.707025
Name: Units Sold, dtype: float64

In [20]:
df["Units Sold"].describe()

count    66900.000000
mean       136.435575
std        109.020346
min          0.000000
25%         48.000000
50%        107.000000
75%        202.000000
max        499.000000
Name: Units Sold, dtype: float64

In [16]:
print(
    "Rows:",
    len(future_forecast)
)

print(
    "Dates:",
    future_forecast["Date"].nunique()
)

print(
    "Stores:",
    future_forecast["Store_ID"].nunique()
)

print(
    "Products:",
    future_forecast["Product_ID"].nunique()
)

print(
    "Min demand:",
    future_forecast["Forecast Demand"].min()
)

print(
    "Max demand:",
    future_forecast["Forecast Demand"].max()
)

print(
    "Total 14-day demand:",
    future_forecast["Forecast Demand"].sum()
)

Rows: 1400
Dates: 14
Stores: 5
Products: 20
Min demand: 29.24737393055079
Max demand: 31.896571154246807
Total 14-day demand: 43805.41726981616


In [17]:
# =========================================================
# SAVE FORECAST
# =========================================================

future_forecast.to_parquet(
    FORECAST_PATH,
    index=False
)

print(
    f"Forecast saved to:\n{FORECAST_PATH}"
)

Forecast saved to:
..\data\processed\forecast\future_forecast_14d.parquet


In [18]:
# =========================================================
# FINAL CHECK
# =========================================================

check_df = pd.read_parquet(
    FORECAST_PATH
)

print(check_df.shape)
print(check_df.head())
print(check_df.tail())

(1400, 4)
        Date Store_ID Product_ID  Forecast Demand
0 2023-11-01     S001      P0001        31.363328
1 2023-11-01     S001      P0002        30.979846
2 2023-11-01     S001      P0003        31.058589
3 2023-11-01     S001      P0004        31.319429
4 2023-11-01     S001      P0005        31.202163
           Date Store_ID Product_ID  Forecast Demand
1395 2023-11-14     S005      P0016        31.363328
1396 2023-11-14     S005      P0017        31.306665
1397 2023-11-14     S005      P0018        29.247374
1398 2023-11-14     S005      P0019        31.426011
1399 2023-11-14     S005      P0020        31.306665
